## Testing with model ONLY trained with brain

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import seaborn as sns

from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from skimage.metrics import structural_similarity as ssim

# Add project root so we can import utils
project_root = Path().cwd().parent.resolve()
sys.path.insert(0, str(project_root))

from src.utils.lu_chipman import lu_chipman_matlab
from src.utils.file_paths import file_paths, TISSUE_DIMENSIONS
from src.utils.visualisation import visualize_save_mueller

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    auc
)

import torch
import torch.nn as nn
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

/Users/chaechae/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
# Add this function after the imports section
def get_sample_sets():
    """
    Returns predefined sample sets for each tissue type
    
    Returns:
        Dictionary with sample sets for each tissue type
    """
    return {
        'brain': {
            '2023-03-22_T_HORAO-91-BF_FR_S1_1_combined.npy',
            '2022-11-22_T_HORAO-67-BF_FR_S1_1_combined.npy', 
            '2024-09-18_T_AUTOPSY-BF_FR_S4_1_combined.npy',
            '2023-05-26_T_HORAO-103-BF_FR_S1_1_combined.npy',
            '2023-05-16_T_HORAO-101-BF_FR_S1_1_combined.npy'
        },
        'cervix': {
            'Sample2_550_PR_combined.npy',
            'Sample18_550_PR_combined.npy',
            'Sample19_550_PR_combined.npy',
            'Sample23_550_PR_combined.npy',
            'Sample25_550_PR_combined.npy'
        },
        'afmmm': {
            'AFMMM_sample_he9_PR_combined.npy',
            'AFMMM_sample_he14_PR_combined.npy',
            'AFMMM_sample_bg5_PR_combined.npy',
            'AFMMM_sample_bw6_PR_combined.npy',
            'AFMMM_sample_bg11_PR_combined.npy'
        }
    }

# Model Definitions

In [3]:
class PixelMLP(nn.Module):
    """MLP Classifier for pixel-wise PR prediction"""
    def __init__(self, in_features=12, hidden_sizes=(128,64,32)):
        super().__init__()
        layers = []
        prev = in_features
        for hsize in hidden_sizes:
            layers += [
                nn.Linear(prev, hsize),
                nn.BatchNorm1d(hsize),
                nn.ReLU(inplace=True),
                nn.Dropout(0.30),
            ]
            prev = hsize
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

### Helper Functions

In [4]:
def predict_mlp(model, X, batch_size=4096, device=None):
    """Generate predictions using MLP model with batching"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            preds.append(probs)
    return np.concatenate(preds)

In [5]:
def compute_ssim(img1, img2, H, W):
    """Compute SSIM between two images with error handling"""
    if img1.size == 0 or img2.size == 0:
        return 0
        
    dr = max(1.0, max(img1.max(), img2.max()) - min(img1.min(), img2.min()))
    # Enforce odd win_size ≤ min(H,W)
    win = max(3, min(7, H if H%2 else H-1, W if W%2 else W-1))
    try:
        return ssim(img1, img2, data_range=dr, win_size=win)
    except Exception as e:
        print(f"SSIM calculation error: {e}")
        return 0


In [6]:
def compute_mueller_metrics(true_matrix, pred_matrix, sample_name):
    """
    Compute metrics between two Mueller matrices
    
    Args:
        true_matrix: Ground truth Mueller matrix (H, W, 4, 4)
        pred_matrix: Predicted Mueller matrix (H, W, 4, 4)
        sample_name: Name of the sample
        
    Returns:
        Dictionary with metrics
    """
    H, W = true_matrix.shape[:2]
    
    # SSIM for all 16 elements
    ssim_dict = {'Sample': sample_name}
    for i in range(4):
        for j in range(4):
            comp = f"M{i+1}{j+1}"
            t_ch = true_matrix[..., i, j]
            p_ch = pred_matrix[..., i, j]
            ssim_dict[comp] = compute_ssim(t_ch, p_ch, H, W)
    
    # Lu-Chipman decomposition
    try:
        # Call lu_chipman_matlab with the matrices in their original shape
        params_true = lu_chipman_matlab(true_matrix)
        params_pred = lu_chipman_matlab(pred_matrix)
        
        # Extract parameters: lu_chipman_matlab returns tuple (D, Delta, LR, CR, psi)
        if isinstance(params_true, tuple) and len(params_true) == 5:
            # Reshape the output parameters to 1D arrays for metric calculation
            diat_true = params_true[0].flatten()  # Diattenuation
            depol_true = params_true[1].flatten()  # Depolarization
            lr_true = params_true[2].flatten()     # Linear retardance
            psi_true = params_true[4].flatten()    # Orientation
            
            diat_pred = params_pred[0].flatten()
            depol_pred = params_pred[1].flatten()
            lr_pred = params_pred[2].flatten()
            psi_pred = params_pred[4].flatten()
            
            # Calculate metrics for each parameter
            decomp_metrics = {}
            
            # Diattenuation metrics
            decomp_metrics["Diattenuation_MSE"] = mean_squared_error(diat_true, diat_pred)
            decomp_metrics["Diattenuation_MAE"] = mean_absolute_error(diat_true, diat_pred)
            decomp_metrics["Diattenuation_R2"] = r2_score(diat_true, diat_pred)
            
            # Depolarization metrics
            decomp_metrics["Depolarization_MSE"] = mean_squared_error(depol_true, depol_pred)
            decomp_metrics["Depolarization_MAE"] = mean_absolute_error(depol_true, depol_pred)
            decomp_metrics["Depolarization_R2"] = r2_score(depol_true, depol_pred)
            
            # Linear Retardance metrics
            decomp_metrics["Retardance_MSE"] = mean_squared_error(lr_true, lr_pred)
            decomp_metrics["Retardance_MAE"] = mean_absolute_error(lr_true, lr_pred)
            decomp_metrics["Retardance_R2"] = r2_score(lr_true, lr_pred)
            
            # Orientation metrics
            decomp_metrics["Orientation_MSE"] = mean_squared_error(psi_true, psi_pred)
            decomp_metrics["Orientation_MAE"] = mean_absolute_error(psi_true, psi_pred)
            decomp_metrics["Orientation_R2"] = r2_score(psi_true, psi_pred)
            
            # Combine all metrics
            all_metrics = {**ssim_dict, **decomp_metrics}
        else:
            print("Unexpected return format from lu_chipman_matlab")
            all_metrics = ssim_dict
            
    except Exception as e:
        print(f"Error in Lu-Chipman decomposition: {e}")
        import traceback
        traceback.print_exc()
        # Return just SSIM metrics if the decomposition fails
        all_metrics = ssim_dict
    
    return all_metrics

### Data Loading Functions

In [7]:
def load_sample_files(tissue_type, sample_set):
    """
    Load sample files for a specific tissue type
    
    Args:
        tissue_type: String indicating tissue type ('brain', 'cervix', or 'afmmm')
        sample_set: Set of sample filenames to use
        
    Returns:
        List of arrays, list of sample names, dimensions
    """
    # Set up file paths based on tissue type
    if tissue_type.lower() == 'brain':
        raw_dir = file_paths.brain_raw_path
        dims = TISSUE_DIMENSIONS['brain']
    elif tissue_type.lower() == 'cervix':
        raw_dir = file_paths.cervix_raw_path
        dims = TISSUE_DIMENSIONS['cervix']
    elif tissue_type.lower() == 'afmmm':
        raw_dir = file_paths.afmmm_raw_path
        dims = TISSUE_DIMENSIONS['afmmm']
    else:
        raise ValueError(f"Invalid tissue type: {tissue_type}")
    
    # Check if directory exists
    assert raw_dir.exists(), f"Raw data directory not found: {raw_dir}"
    
    # Find sample files
    iso_files = [p for p in raw_dir.glob('**/*.npy') if p.name in sample_set]
    print(f"Found {len(iso_files)} {tissue_type} files for evaluation.")
    
    if len(iso_files) == 0:
        print(f"Warning: No {tissue_type} samples found in {raw_dir}")
        return [], [], None
    
    # Load sample files
    arrays = []
    sample_names = []
    
    for fp in tqdm(iso_files, desc=f"Loading {tissue_type} samples"):
        sample_names.append(fp.stem.replace('_combined', ''))
        try:
            arr = np.load(fp)
            # Verify shape
            expected = (dims['num_rows'], dims['num_cols'], 17)
            if arr.shape != expected:
                print(f"Warning: {fp.name} shape {arr.shape} != {expected}")
                continue
            arrays.append(arr)
        except Exception as e:
            print(f"Failed to load {fp.name}: {e}")
    
    return arrays, sample_names, dims

In [8]:
def merge_and_prepare_data(arrays):
    """
    Merge arrays and prepare data for model input
    
    Args:
        arrays: List of arrays to merge
        
    Returns:
        X_flat: Flattened features (N, 12)
        y_flat: Flattened labels (N,)
        merged: Merged array (ns, h, w, 17)
    """
    if not arrays:
        return None, None, None
    
    # Merge into one array
    merged = np.stack(arrays, axis=0)
    print(f"Merged shape:", merged.shape)
    
    # Extract features and labels
    X = merged[..., :12]
    y = merged[..., -1]
    
    # Flatten to per-pixel
    ns, h, w, c = X.shape
    X_flat = X.reshape(-1, c)
    y_flat = y.reshape(-1)
    
    return X_flat, y_flat, merged

### Model Loading and Prediction Functions

In [9]:
def load_and_predict(X_flat, use_model='xgb'):
    """
    Load ML model and make predictions
    
    Args:
        X_flat: Flattened features (N, 12)
        use_model: ML model to use for PR prediction ('mlp', 'xgb', or 'cat')
        
    Returns:
        y_pred: Predicted labels (N,)
        probs: Predicted probabilities (N,)
    """
    # Device for PyTorch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load models
    if use_model.lower() == 'mlp':
        # MLP model
        mlp = PixelMLP().to(device)
        mlp_path = file_paths.model_save_path /'brain'/'best_pixel_mlp.pth'
        mlp.load_state_dict(torch.load(mlp_path, map_location=device))
        mlp.eval()
        # Get predictions
        probs = predict_mlp(mlp, X_flat, device=device)
        y_pred = (probs > 0.5).astype(int)
        print("Using MLP model for PR prediction")
    elif use_model.lower() == 'xgb':
        # XGBoost
        xgb_model = xgb.Booster()
        xgb_model.load_model(str(file_paths.model_save_path /'brain'/ 'pixel_xgb.json'))
        # Get predictions
        dmat_iso = xgb.DMatrix(X_flat)
        probs = xgb_model.predict(dmat_iso)
        y_pred = (probs > 0.5).astype(int)
        print("Using XGBoost model for PR prediction")
    elif use_model.lower() == 'cat':
        # CatBoost
        cat_model = CatBoostClassifier()
        cat_model.load_model(str(file_paths.model_save_path /'brain'/ 'pixel_catboost.cbm'))
        # Get predictions
        probs = cat_model.predict_proba(X_flat)[:,1]
        y_pred = (probs > 0.5).astype(int)
        print("Using CatBoost model for PR prediction")
    else:
        raise ValueError(f"Invalid model type: {use_model}")
    
    return y_pred, probs

In [10]:
def compute_classification_metrics(y_true, y_pred, probs):
    """
    Compute classification metrics
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        probs: Predicted probabilities
        
    Returns:
        Dictionary with metrics
    """
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, probs)
    
    metrics = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc
    }
    
    print(f"Classification Metrics:")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}, ROC AUC: {roc:.4f}")
    
    return metrics

### Last Row MM Generation Functions

In [11]:
def generate_method_a_matrix(arr, H, W):
    """
    Generate Mueller matrix using Method A: Filtered data using original PR mask (ALL elements)
    
    Args:
        arr: Sample array (H, W, 17)
        H: Height of the array
        W: Width of the array
        
    Returns:
        method_a_matrix: Mueller matrix (H, W, 4, 4)
    """
    n_pix = H * W
    X_flat = arr[..., :12].reshape(n_pix, 12)
    true_last = arr[..., 12:16].reshape(n_pix, 4)
    original_mask = arr[..., -1].reshape(n_pix)
    
    method_a_mueller = np.zeros((n_pix, 16))
    
    # Only PR pixels retain their values
    pr_indices = np.where(original_mask == 1)[0]
    if len(pr_indices) > 0:
        method_a_mueller[pr_indices, :12] = X_flat[pr_indices, :]
        method_a_mueller[pr_indices, 12:16] = true_last[pr_indices]
    
    # Reshape to (H, W, 4, 4)
    return method_a_mueller.reshape(H, W, 4, 4), pr_indices


In [12]:
def generate_method_b_matrix(arr, H, W, v2_model):
    """
    Generate Mueller matrix using Method B: Original PR mask with filtered 12 elements and generated last row
    
    Args:
        arr: Sample array (H, W, 17)
        H: Height of the array
        W: Width of the array
        v2_model: Model for last row generation
        
    Returns:
        method_b_matrix: Mueller matrix (H, W, 4, 4)
    """
    n_pix = H * W
    X_flat = arr[..., :12].reshape(n_pix, 12)
    original_mask = arr[..., -1].reshape(n_pix)
    
    method_b_mueller = np.zeros((n_pix, 16))
    
    # Only PR pixels retain their values for first 12 elements
    pr_indices = np.where(original_mask == 1)[0]
    if len(pr_indices) > 0:
        method_b_mueller[pr_indices, :12] = X_flat[pr_indices, :]
        
        # Generate last row for PR pixels
        generated_last_row_orig_pr = v2_model.predict(X_flat[pr_indices])
        method_b_mueller[pr_indices, 12:16] = generated_last_row_orig_pr
    
    # Reshape to (H, W, 4, 4)
    return method_b_mueller.reshape(H, W, 4, 4)

In [13]:
def generate_method_c_matrix(arr, H, W, v2_model, ml_mask):
    """
    Generate Mueller matrix using Method C: ML PR mask with filtered first 12 elements and generated last row
    
    Args:
        arr: Sample array (H, W, 17)
        H: Height of the array
        W: Width of the array
        v2_model: Model for last row generation
        ml_mask: ML-predicted PR mask
        
    Returns:
        method_c_matrix: Mueller matrix (H, W, 4, 4)
    """
    n_pix = H * W
    X_flat = arr[..., :12].reshape(n_pix, 12)
    
    method_c_mueller = np.zeros((n_pix, 16))
    
    # Only ML-predicted PR pixels retain their values
    ml_pr_indices = np.where(ml_mask == 1)[0]
    if len(ml_pr_indices) > 0:
        method_c_mueller[ml_pr_indices, :12] = X_flat[ml_pr_indices, :]
        
        # Generate last row for ML-PR pixels
        generated_last_row = v2_model.predict(X_flat[ml_pr_indices])
        method_c_mueller[ml_pr_indices, 12:16] = generated_last_row
    
    # Reshape to (H, W, 4, 4)
    return method_c_mueller.reshape(H, W, 4, 4), ml_pr_indices

### Comparison and Visualization Functions

In [14]:
def compute_direct_comparisons(method_a_matrix, method_b_matrix, method_c_matrix, sample_name):
    """
    Compute direct SSIM comparisons between methods
    
    Args:
        method_a_matrix: Mueller matrix from Method A (H, W, 4, 4)
        method_b_matrix: Mueller matrix from Method B (H, W, 4, 4)
        method_c_matrix: Mueller matrix from Method C (H, W, 4, 4)
        sample_name: Name of the sample
        
    Returns:
        Dictionary with comparison results
    """
    H, W = method_a_matrix.shape[:2]
    
    a_vs_b_ssim = {'Sample': f"{sample_name}_A_vs_B"}
    a_vs_c_ssim = {'Sample': f"{sample_name}_A_vs_C"}
    b_vs_c_ssim = {'Sample': f"{sample_name}_B_vs_C"}
    
    # Compute SSIM for all 16 elements for each comparison
    for r in range(4):
        for c in range(4):
            element = f"M{r+1}{c+1}"
            
            # A vs B
            a_ch = method_a_matrix[..., r, c]
            b_ch = method_b_matrix[..., r, c]
            a_vs_b_ssim[element] = compute_ssim(a_ch, b_ch, H, W)
            
            # A vs C
            c_ch = method_c_matrix[..., r, c]
            a_vs_c_ssim[element] = compute_ssim(a_ch, c_ch, H, W)
            
            # B vs C
            b_vs_c_ssim[element] = compute_ssim(b_ch, c_ch, H, W)
    
    return a_vs_b_ssim, a_vs_c_ssim, b_vs_c_ssim


In [15]:
def save_visualizations(method_a_mueller, method_b_mueller, method_c_mueller, H, W, sample_name, tissue_type):
    """
    Save visualizations of Mueller matrices
    
    Args:
        method_a_mueller: Mueller matrix from Method A (flattened)
        method_b_mueller: Mueller matrix from Method B (flattened)
        method_c_mueller: Mueller matrix from Method C (flattened)
        H: Height of the array
        W: Width of the array
        sample_name: Name of the sample
        tissue_type: Type of tissue
    """
    try:
        # Convert to DataFrame for visualization
        df_method_a = pd.DataFrame(method_a_mueller)
        df_method_b = pd.DataFrame(method_b_mueller)
        df_method_c = pd.DataFrame(method_c_mueller)
        
        # Save visualizations
        viz_path = file_paths.figures / tissue_type
        viz_path.mkdir(exist_ok=True, parents=True)
        
        visualize_save_mueller(
            data=df_method_a,
            visualisation_path=viz_path,
            sample_number=f"{sample_name}_methodA_filtered_original_PR",
            wavelength=550,
            num_rows=H, num_cols=W
        )
        
        visualize_save_mueller(
            data=df_method_b,
            visualisation_path=viz_path,
            sample_number=f"{sample_name}_methodB_original_PR_gen_row",
            wavelength=550,
            num_rows=H, num_cols=W
        )
        
        visualize_save_mueller(
            data=df_method_c,
            visualisation_path=viz_path,
            sample_number=f"{sample_name}_methodC_ML_PR_gen_row",
            wavelength=550,
            num_rows=H, num_cols=W
        )
        
        print(f"Visualizations saved for sample {sample_name}")
    except Exception as e:
        print(f"Error in visualization: {e}")

In [16]:
def analyze_comparison_results(df_a_vs_b, df_a_vs_c, df_b_vs_c, tissue_type, model_type):
    """
    Analyze comparison results and create visualizations
    
    Args:
        df_a_vs_b: DataFrame with A vs B comparison results
        df_a_vs_c: DataFrame with A vs C comparison results
        df_b_vs_c: DataFrame with B vs C comparison results
        tissue_type: Type of tissue
        model_type: Type of ML model used
        
    Returns:
        DataFrame with average comparison metrics
    """
    # Identify SSIM columns
    numeric_cols = df_a_vs_b.select_dtypes(include=[np.number]).columns
    ssim_cols = [col for col in numeric_cols if col.startswith('M')]
    
    # Calculate mean SSIM values
    avg_a_vs_b = df_a_vs_b[ssim_cols].mean()
    avg_a_vs_c = df_a_vs_c[ssim_cols].mean()
    avg_b_vs_c = df_b_vs_c[ssim_cols].mean()
    
    # Create comparison DataFrame
    comparison_df = pd.DataFrame({
        'A_vs_B': avg_a_vs_b,
        'A_vs_C': avg_a_vs_c,
        'B_vs_C': avg_b_vs_c
    })
    
    # Save comparison results
    out_dir = file_paths.results / tissue_type
    out_dir.mkdir(exist_ok=True, parents=True)
    comparison_df.to_csv(out_dir / f'comparison_summary_{model_type}.csv')
    
    # Visualize comparison results
    plt.figure(figsize=(15, 10))
    comparison_df.plot(kind='bar', ax=plt.gca())
    plt.title(f'{tissue_type} - Direct SSIM Comparison Between Methods ({model_type})')
    plt.ylabel('SSIM Value')
    plt.xlabel('Mueller Matrix Element')
    plt.xticks(rotation=45)
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(out_dir / f'direct_ssim_comparison_{model_type}.png', dpi=300, bbox_inches='tight')
    
    return comparison_df


### Main Evaluation Functions

In [17]:
def process_single_sample(arr, sample_name, v2_model, sample_ml_mask, tissue_type=None, visualize=False):
    """
    Process a single sample to generate Mueller matrices for all methods
    
    Args:x
        arr: Sample array (H, W, 17)
        sample_name: Name of the sample
        v2_model: Model for last row generation
        sample_ml_mask: ML-predicted PR mask
        tissue_type: Type of tissue (for visualization)
        visualize: Whether to save visualizations
        
    Returns:
        Dictionary with results for this sample
    """
    H, W, _ = arr.shape
    n_pix = H * W
    
    # Extract components
    X_flat = arr[..., :12].reshape(n_pix, 12)
    true_last = arr[..., 12:16].reshape(n_pix, 4)
    original_mask = arr[..., -1].reshape(n_pix)
    
    # Method A: Original PR with original last row
    method_a_matrix, pr_indices = generate_method_a_matrix(arr, H, W)
    
    # Method B: Original PR with generated last row
    method_b_matrix = generate_method_b_matrix(arr, H, W, v2_model)
    
    # Method C: ML PR with generated last row
    method_c_matrix, ml_pr_indices = generate_method_c_matrix(arr, H, W, v2_model, sample_ml_mask)
    
    # Print basic stats
    print(f"\nSample {sample_name}:")
    print(f"  Original PR pixels: {len(pr_indices)} ({len(pr_indices)/n_pix*100:.2f}%)")
    print(f"  ML PR pixels: {len(ml_pr_indices)} ({len(ml_pr_indices)/n_pix*100:.2f}%)")
    
    # Compute direct comparisons
    a_vs_b_ssim, a_vs_c_ssim, b_vs_c_ssim = compute_direct_comparisons(
        method_a_matrix, method_b_matrix, method_c_matrix, sample_name
    )
    
    # Save visualizations if requested
    if visualize and tissue_type:
        # Convert matrices back to flattened for visualization
        method_a_mueller = method_a_matrix.reshape(n_pix, 16)
        method_b_mueller = method_b_matrix.reshape(n_pix, 16)
        method_c_mueller = method_c_matrix.reshape(n_pix, 16)
        
        save_visualizations(
            method_a_mueller, method_b_mueller, method_c_mueller,
            H, W, sample_name, tissue_type
        )
    
    return {
        'sample_name': sample_name,
        'method_a_matrix': method_a_matrix,
        'method_b_matrix': method_b_matrix,
        'method_c_matrix': method_c_matrix,
        'a_vs_b_ssim': a_vs_b_ssim,
        'a_vs_c_ssim': a_vs_c_ssim,
        'b_vs_c_ssim': b_vs_c_ssim,
        'pr_indices': pr_indices,
        'ml_pr_indices': ml_pr_indices
    }

In [18]:
def evaluate_tissue_samples(tissue_type, sample_set, use_model="xgb"):
    """
    Run evaluation workflow for a specific tissue type
    
    Args:
        tissue_type: String indicating tissue type ('brain', 'cervix', or 'afmmm')
        sample_set: Set of sample filenames to use
        use_model: ML model to use for PR prediction ('mlp', 'xgb', or 'cat')
        
    Returns:
        Dictionary with evaluation results
    """
    print(f"\n{'='*20} Evaluating {tissue_type} samples {'='*20}")
    
    # 1. Load sample files
    arrays, sample_names, dims = load_sample_files(tissue_type, sample_set)
    
    if not arrays:
        return None
    
    # 2. Prepare data
    X_flat, y_flat, iso_merged = merge_and_prepare_data(arrays)
    
    # 3. Load models and predict
    y_pred, probs = load_and_predict(X_flat, use_model=use_model)
    
    # 4. Compute classification metrics
    metrics = compute_classification_metrics(y_flat, y_pred, probs)
    
    # 5. Load last-row generation model
    v2_model = load_model(file_paths.model_save_path / 'nn_model_v2b.keras')
    
    # 6. Reshape predictions back to image dimensions
    ns, h, w, _ = iso_merged.shape
    ml_mask = y_pred.reshape(ns, h, w)
    
    # 7. Process each sample
    method_a_matrices = []
    method_b_matrices = []
    method_c_matrices = []
    a_vs_b_results = []
    a_vs_c_results = []
    b_vs_c_results = []
    
    # Processing samples loop
    bar_fmt = "{l_bar}{bar}  {n_fmt}/{total_fmt}  [{elapsed}<{remaining}]"
    for idx in tqdm(range(len(arrays)), desc=f"Processing {tissue_type} samples", leave=False, bar_format=bar_fmt):
        # Get sample data
        sample_name = sample_names[idx]
        arr = iso_merged[idx]
        sample_ml_mask = ml_mask[idx].reshape(-1)
        
        # Only visualize first 2 samples to save time
        visualize = (idx < 2)
        
        # Process the sample
        result = process_single_sample(
            arr, sample_name, v2_model, sample_ml_mask, 
            tissue_type=tissue_type, visualize=visualize
        )
        
        # Store results
        method_a_matrices.append(result['method_a_matrix'])
        method_b_matrices.append(result['method_b_matrix'])
        method_c_matrices.append(result['method_c_matrix'])
        
        # Store comparison results
        a_vs_b_results.append(result['a_vs_b_ssim'])
        a_vs_c_results.append(result['a_vs_c_ssim'])
        b_vs_c_results.append(result['b_vs_c_ssim'])
    
    # 8. Convert comparison results to DataFrames
    df_a_vs_b = pd.DataFrame(a_vs_b_results)
    df_a_vs_c = pd.DataFrame(a_vs_c_results)
    df_b_vs_c = pd.DataFrame(b_vs_c_results)
    
    # 9. Save results
    out_dir = file_paths.results / tissue_type
    out_dir.mkdir(exist_ok=True, parents=True)
    
    df_a_vs_b.to_csv(out_dir / f'method_a_vs_b_ssim_{use_model}.csv', index=False)
    df_a_vs_c.to_csv(out_dir / f'method_a_vs_c_ssim_{use_model}.csv', index=False)
    df_b_vs_c.to_csv(out_dir / f'method_b_vs_c_ssim_{use_model}.csv', index=False)
    
    # 10. Analyze comparison results
    comparison_df = analyze_comparison_results(df_a_vs_b, df_a_vs_c, df_b_vs_c, tissue_type, use_model)
    
    print(f"\n{tissue_type} evaluation complete. Results saved to {out_dir}")
    
    return {
        'tissue_type': tissue_type,
        'classification_metrics': metrics,
        'comparison_metrics': comparison_df,
        'matrices': {
            'method_a': method_a_matrices,
            'method_b': method_b_matrices,
            'method_c': method_c_matrices
        },
        'comparison_results': {
            'a_vs_b': df_a_vs_b,
            'a_vs_c': df_a_vs_c,
            'b_vs_c': df_b_vs_c
        }
    }

In [19]:
def create_summary_comparison(results, model_type='xgb'):
    """
    Create summary comparison across all tissue types
    
    Args:
        results: Dictionary with results for all tissue types
        model_type: Type of ML model used
        
    Returns:
        DataFrames with summary comparisons
    """
    if not results:
        print("No results to compare.")
        return None, None
    
    # Create summary directory
    summary_dir = file_paths.results / 'summary'
    summary_dir.mkdir(exist_ok=True, parents=True)
    
    # Classification metrics summary
    class_metrics = {
        tissue: {
            'Accuracy': results[tissue]['classification_metrics']['accuracy'],
            'Precision': results[tissue]['classification_metrics']['precision'],
            'Recall': results[tissue]['classification_metrics']['recall'],
            'F1': results[tissue]['classification_metrics']['f1'],
            'ROC_AUC': results[tissue]['classification_metrics']['roc_auc']
        } for tissue in results
    }
    
    df_class = pd.DataFrame(class_metrics).T
    df_class.to_csv(summary_dir / f'classification_metrics_summary_{model_type}.csv')
    
    print("\n===== Classification Metrics Summary =====")
    print(df_class)
    
    # Create comparison metrics summary
    comparison_summary = {}
    
    for tissue in results:
        # Calculate average SSIM across all matrix elements
        comp_df = results[tissue]['comparison_metrics']
        comparison_summary[tissue] = {
            'A_vs_B_avg': comp_df['A_vs_B'].mean(),
            'A_vs_C_avg': comp_df['A_vs_C'].mean(),
            'B_vs_C_avg': comp_df['B_vs_C'].mean()
        }
    
    df_comp = pd.DataFrame(comparison_summary)
    df_comp = df_comp.T  # Transpose for better readability
    df_comp.to_csv(summary_dir / f'comparison_metrics_summary_{model_type}.csv')
    
    print("\n===== Comparison Metrics Summary =====")
    print(df_comp)
    
    # Create visualization for summary comparison
    plt.figure(figsize=(12, 6))
    df_comp.plot(kind='bar', ax=plt.gca())
    plt.title(f'Average SSIM Comparison Across Tissue Types ({model_type})')
    plt.ylabel('Average SSIM Value')
    plt.xlabel('Tissue Type')
    plt.xticks(rotation=0)
    plt.legend(title='Comparison')
    plt.tight_layout()
    plt.savefig(summary_dir / f'comparison_summary_{model_type}.png', dpi=300, bbox_inches='tight')
    
    return df_class, df_comp

In [20]:
def run_all_evaluations(model_type='xgb'):
    """
    Run evaluation on all sample types
    
    Args:
        model_type: Model to use for PR prediction ('mlp', 'xgb', or 'cat')
        
    Returns:
        Dictionary with results for all tissue types
    """
    # Define sample sets
    brain_samples = {
        '2023-03-22_T_HORAO-91-BF_FR_S1_1_combined.npy',
        '2022-11-22_T_HORAO-67-BF_FR_S1_1_combined.npy', 
        '2024-09-18_T_AUTOPSY-BF_FR_S4_1_combined.npy',
        '2023-05-26_T_HORAO-103-BF_FR_S1_1_combined.npy',
        '2023-05-16_T_HORAO-101-BF_FR_S1_1_combined.npy'
    }

    cervix_samples = {
        'Sample2_PR_combined.npy',
        'Sample18_PR_combined.npy',
        'Sample19_PR_combined.npy',
        'Sample23_PR_combined.npy',
        'Sample25_PR_combined.npy'
    }

    afmmm_samples = {
        'Sample2_PR_combined.npy',
        'Sample18_PR_combined.npy',
        'Sample19_PR_combined.npy',
        'Sample23_PR_combined.npy',
        'Sample25_PR_combined.npy'
    }
    
    # Run evaluations
    results = {}
    
    # Evaluate brain samples
    brain_results = evaluate_tissue_samples('brain', brain_samples, use_model=model_type)
    if brain_results is not None:
        results['brain'] = brain_results
    
    # Evaluate cervix samples
    cervix_results = evaluate_tissue_samples('cervix', cervix_samples, use_model=model_type)
    if cervix_results is not None:
        results['cervix'] = cervix_results
    
    # Evaluate AFMMM samples
    afmmm_results = evaluate_tissue_samples('afmmm', afmmm_samples, use_model=model_type)
    if afmmm_results is not None:
        results['afmmm'] = afmmm_results
    
    # Create summary comparisons
    if results:
        df_class, df_comp = create_summary_comparison(results, model_type)
    
    return results

In [21]:
def compare_ml_models(tissue_type='brain', sample_set=None):
    """
    Compare all ML models on a specific tissue type
    
    Args:
        tissue_type: Tissue type to evaluate
        sample_set: Set of sample filenames to use (if None, uses default brain samples)
        
    Returns:
        Dictionary with results for all models
    """
    if sample_set is None:
        # Default to brain samples
        sample_set = {
            '2023-03-22_T_HORAO-91-BF_FR_S1_1_combined.npy',
            '2022-11-22_T_HORAO-67-BF_FR_S1_1_combined.npy', 
            '2024-09-18_T_AUTOPSY-BF_FR_S4_1_combined.npy',
            '2023-05-26_T_HORAO-103-BF_FR_S1_1_combined.npy',
            '2023-05-16_T_HORAO-101-BF_FR_S1_1_combined.npy'
        }
    
    print(f"\n{'='*20} Comparing ML models on {tissue_type} samples {'='*20}")
    
    # Run evaluation with MLP
    print("\n--- Evaluating with MLP ---")
    mlp_results = evaluate_tissue_samples(tissue_type, sample_set, use_model='mlp')
    
    # Run evaluation with XGBoost
    print("\n--- Evaluating with XGBoost ---")
    xgb_results = evaluate_tissue_samples(tissue_type, sample_set, use_model='xgb')
    
    # Run evaluation with CatBoost
    print("\n--- Evaluating with CatBoost ---")
    cat_results = evaluate_tissue_samples(tissue_type, sample_set, use_model='cat')
    
    # Compare results
    print("\n=== Model Comparison Results ===")
    if all(x is not None for x in [mlp_results, xgb_results, cat_results]):
        # Create comparison DataFrame for classification metrics
        class_metrics = {
            'MLP': mlp_results['classification_metrics'],
            'XGBoost': xgb_results['classification_metrics'],
            'CatBoost': cat_results['classification_metrics']
        }
        
        df_class = pd.DataFrame(class_metrics)
        print("\nClassification Metrics Comparison:")
        print(df_class)
        
        # Save comparison results
        out_dir = file_paths.results / 'model_comparison' / tissue_type
        out_dir.mkdir(exist_ok=True, parents=True)
        df_class.to_csv(out_dir / 'classification_metrics_comparison.csv')
        
        # Create visualization
        plt.figure(figsize=(12, 8))
        df_class.plot(kind='bar', ax=plt.gca())
        plt.title(f'Classification Metrics Comparison Across Models ({tissue_type})')
        plt.ylabel('Score')
        plt.xticks(rotation=0)
        plt.legend(title='Model')
        plt.tight_layout()
        plt.savefig(out_dir / 'model_comparison.png', dpi=300, bbox_inches='tight')
        print(f"\nComparison results saved to {out_dir}")
        
        # Also compare SSIM metrics
        avg_ssim_metrics = {
            'MLP': {
                'A_vs_B': mlp_results['comparison_metrics']['A_vs_B'].mean(),
                'A_vs_C': mlp_results['comparison_metrics']['A_vs_C'].mean(),
                'B_vs_C': mlp_results['comparison_metrics']['B_vs_C'].mean()
            },
            'XGBoost': {
                'A_vs_B': xgb_results['comparison_metrics']['A_vs_B'].mean(),
                'A_vs_C': xgb_results['comparison_metrics']['A_vs_C'].mean(),
                'B_vs_C': xgb_results['comparison_metrics']['B_vs_C'].mean()
            },
            'CatBoost': {
                'A_vs_B': cat_results['comparison_metrics']['A_vs_B'].mean(),
                'A_vs_C': cat_results['comparison_metrics']['A_vs_C'].mean(),
                'B_vs_C': cat_results['comparison_metrics']['B_vs_C'].mean()
            }
        }
        
        df_ssim = pd.DataFrame(avg_ssim_metrics)
        print("\nAverage SSIM Comparison Across Models:")
        print(df_ssim)
        
        df_ssim.to_csv(out_dir / 'ssim_metrics_comparison.csv')
        
        # Create visualization
        plt.figure(figsize=(12, 8))
        df_ssim.plot(kind='bar', ax=plt.gca())
        plt.title(f'Average SSIM Comparison Across Models ({tissue_type})')
        plt.ylabel('Average SSIM')
        plt.xticks(rotation=0)
        plt.legend(title='Model')
        plt.tight_layout()
        plt.savefig(out_dir / 'ssim_comparison.png', dpi=300, bbox_inches='tight')
        
        return {
            'mlp': mlp_results,
            'xgb': xgb_results,
            'cat': cat_results,
            'classification_comparison': df_class,
            'ssim_comparison': df_ssim
        }
    else:
        print("One or more models failed to evaluate.")
        return None

In [22]:
def run_notebook_demo(tissue_type='brain', model_type='xgb', visualize_samples=True):
    """
    Demonstration function designed for notebook environments
    
    Args:
        tissue_type: Tissue type to evaluate ('brain', 'cervix', or 'afmmm')
        model_type: Model to use ('mlp', 'xgb', or 'cat')
        visualize_samples: Whether to visualize sample results
        
    Returns:
        Evaluation results
    """
    # Get sample sets
    sample_sets = get_sample_sets()
    
    if tissue_type not in sample_sets:
        print(f"Invalid tissue type: {tissue_type}")
        print(f"Available tissue types: {list(sample_sets.keys())}")
        return None
    
    print(f"Running evaluation on {tissue_type} samples with {model_type} model...")
    
    # Run evaluation
    results = evaluate_tissue_samples(tissue_type, sample_sets[tissue_type], use_model=model_type)
    
    # Visualize some results if requested
    if visualize_samples and results is not None:
        # Display a summary of the results
        print("\nClassification Metrics:")
        for metric, value in results['classification_metrics'].items():
            print(f"  {metric}: {value:.4f}")
        
        # Display SSIM comparison
        comparison_df = results['comparison_metrics']
        
        import matplotlib.pyplot as plt
        plt.figure(figsize=(12, 6))
        comparison_df.plot(kind='bar', ax=plt.gca())
        plt.title(f'{tissue_type} - Direct SSIM Comparison Between Methods ({model_type})')
        plt.ylabel('SSIM Value')
        plt.xlabel('Mueller Matrix Element')
        plt.xticks(rotation=45)
        plt.legend(loc='lower right', fontsize=10)
        plt.tight_layout()
        plt.show()
    
    return results

In [23]:
if __name__ == "__main__":
    # Check if we're in a Jupyter environment
    def is_jupyter():
        try:
            get_ipython()
            return True
        except NameError:
            return False
    
    # If in Jupyter, provide examples and helper functions
    if is_jupyter():
        print("Running in Jupyter notebook mode")
        print("To evaluate samples, use one of these functions:")
        print("1. results = evaluate_tissue_samples('brain', get_sample_sets()['brain'], use_model='xgb')")
        print("2. results = run_all_evaluations(model_type='xgb')")
        print("3. results = compare_ml_models('brain', get_sample_sets()['brain'])")
        print("\nOr use the notebook demo function:")
        print("results = run_notebook_demo(tissue_type='brain', model_type='xgb')")
    
    # If not in Jupyter, use command-line arguments
    else:
        import argparse
        parser = argparse.ArgumentParser(description='Run Mueller Matrix evaluation on multiple tissue types')
        parser.add_argument('--model', type=str, choices=['mlp', 'xgb', 'cat'], default='xgb',
                           help='Model to use for PR prediction (default: xgb)')
        parser.add_argument('--tissue', type=str, choices=['brain', 'cervix', 'afmmm', 'all'], default='all',
                           help='Tissue type to evaluate (default: all)')
        parser.add_argument('--compare-models', action='store_true',
                           help='Compare all ML models on the specified tissue type')
        
        args = parser.parse_args()
        
        # Setup
        print(f"Using {args.model} model for evaluation")
        
        # Get sample sets
        sample_sets = get_sample_sets()
        
        if args.compare_models:
            # Compare all models on the specified tissue type
            if args.tissue == 'all':
                print("Cannot compare models on all tissue types. Please specify a single tissue type.")
                sys.exit(1)
            
            compare_ml_models(args.tissue, sample_sets[args.tissue])
        elif args.tissue == 'all':
            # Run all evaluations
            results = run_all_evaluations(model_type=args.model)
        else:
            # Run single tissue evaluation
            evaluate_tissue_samples(args.tissue, sample_sets[args.tissue], use_model=args.model)
        
        print("\nEvaluation complete!")

Running in Jupyter notebook mode
To evaluate samples, use one of these functions:
1. results = evaluate_tissue_samples('brain', get_sample_sets()['brain'], use_model='xgb')
2. results = run_all_evaluations(model_type='xgb')
3. results = compare_ml_models('brain', get_sample_sets()['brain'])

Or use the notebook demo function:
results = run_notebook_demo(tissue_type='brain', model_type='xgb')


In [24]:
results = run_all_evaluations(model_type='xgb')


==================== Evaluating brain samples ====================
Found 5 brain files for evaluation.


Loading brain samples:   0%|          | 0/5 [00:00<?, ?it/s]

Merged shape: (5, 388, 516, 17)


XGBoostError: [14:47:46] /Users/runner/work/xgboost/xgboost/src/common/io.cc:146: Opening /Users/chaechae/Desktop/EP_Code/pr_prediction/model/brain/pixel_xgb.json failed: No such file or directory
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x000000038af8dbfc dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x000000038b07b92c xgboost::common::LoadSequentialFile(std::__1::basic_string<char, std::__1::char_traits<char>, std::__1::allocator<char>>) + 728
  [bt] (2) 3   libxgboost.dylib                    0x000000038b016dd0 XGBoosterLoadModel::$_4::operator()() const + 144
  [bt] (3) 4   libxgboost.dylib                    0x000000038b0169d0 XGBoosterLoadModel + 620
  [bt] (4) 5   libffi.8.dylib                      0x0000000101e1404c ffi_call_SYSV + 76
  [bt] (5) 6   libffi.8.dylib                      0x0000000101e11834 ffi_call_int + 1404
  [bt] (6) 7   _ctypes.cpython-311-darwin.so       0x0000000101df4150 _ctypes_ca

In [ ]:
# To see brain results
brain_results = results['brain']

In [ ]:
# For cervix tissue only
cervix_results = evaluate_tissue_samples('cervix', get_sample_sets()['cervix'], use_model='xgb')



In [51]:
# For AFMMM tissue only
afmmm_results = evaluate_tissue_samples('afmmm', get_sample_sets()['afmmm'], use_model='xgb')


==================== Evaluating afmmm samples ====================


AttributeError: 'FilePath' object has no attribute 'afmmm_raw_path'